# Ablation 2026-09-01 — Flow Matching vs Diffusion: Inference Steps Study

Side-by-side comparison of sample quality vs. number of function evaluations (NFE) for:
- **FlowMatching** (Euler ODE): 5, 10, 25, 50, 100 steps
- **LDM / DiffusionSR** (DDIM): skip=1 (1000 NFE), 5 (200), 10 (100), 20 (50), 50 (20)

Shows the key advantage of flow matching: high quality in far fewer NFEs.

Load models from the ablation_20260901 runs (enc+sdf variants recommended as they are the most capable).


In [ ]:
# ── USER CONFIG ────────────────────────────────────────────────────────────────
# FlowMatching run (enc+sdf recommended)
FM_RUN_DIR  = '/scratch/ngng/runs/abl_fm_enc_sdf'
FM_ENC_DIR  = '/trace/group/forgelab/ngng/multifield/DiffusionSR_shohom/diffusionsr/runs/ablation_20260901/enc_sdf'
FM_ENCODING = True; FM_COND = 'implicit'

# DDPM/LDM run — use LDM enc+sdf from ablation OR existing DiffusionSR from nb08
# For LDM: set MODEL2='ldm', provide VAE_DIR
# For DiffusionSR: set MODEL2='diffusion', leave VAE_DIR=None
MODEL2      = 'ldm'  # 'ldm' or 'diffusion'
DIFF_RUN_DIR = '/scratch/ngng/runs/abl_ldm_enc_sdf'
DIFF_ENC_DIR = '/trace/group/forgelab/ngng/multifield/DiffusionSR_shohom/diffusionsr/runs/ablation_20260901/enc_sdf'
VAE_DIR      = '/trace/group/forgelab/ngng/multifield/DiffusionSR_shohom/diffusionsr/runs/ablation_20260901/vae_sdf'
DIFF_ENCODING = True; DIFF_COND = 'implicit'

DATA_ROOT        = '/trace/group/forgelab/ngng/multifield/data_fields'
FIELD_NAMES      = ['temperature', 'sdfliqlabel']
N_STEPS          = 3; DOWNSCALE_METHOD = 'direct'; NORMALIZE = 'standardize'
TIMESTEPS        = 1000; SCHEDULE = 'linear'
DEVICE           = 'cuda'
ANALYSIS_CH      = 0; BATCH_SIZE = 4; N_ABL_BATCHES = 4

# Ablation grids
FM_EULER_STEPS  = [5, 10, 25, 50, 100]   # Euler ODE steps (NFE for FlowMatching)
DDIM_SKIPS      = [50, 20, 10, 5, 1]     # DDIM skip; NFE = TIMESTEPS/skip

EVAL_OUT_DIR = '/trace/group/forgelab/ngng/multifield/eval_results/ablation_20260901/steps_study'

In [ ]:
%matplotlib inline
import os,sys,time; from pathlib import Path
import numpy as np,torch,pandas as pd; import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from IPython.display import display as _ipy_display
plt.show=lambda *a,**kw:[_ipy_display(plt.figure(n)) for n in plt.get_fignums()] or plt.close('all')
if not torch.cuda.is_available() and DEVICE=='cuda': DEVICE='cpu'; print('CPU fallback')
def find_root(s=Path.cwd()):
    for p in [s,*s.parents]:
        if (p/'setup.py').exists() and (p/'diffusionsr').exists(): return p
    raise RuntimeError('no root')
PROJECT_ROOT=find_root()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0,str(PROJECT_ROOT))
from diffusionsr.datasets.dataset import SimulationXZDataset
from diffusionsr.analysis.analysis_functions import get_profile
def as_numpy(x): return x.detach().cpu().numpy() if isinstance(x,torch.Tensor) else np.asarray(x)
def mae_rmse(p,g): p,g=np.asarray(p).ravel(),np.asarray(g).ravel(); return float(np.mean(np.abs(p-g)))
kw=dict(downscale_method=DOWNSCALE_METHOD,root_folder=DATA_ROOT,normalize=NORMALIZE,n_steps=N_STEPS,field_names=FIELD_NAMES)
train_ds,dev_ds,test_ds=(SimulationXZDataset(split=s,**kw) for s in ['train','dev','test'])
print(f'Fields:{train_ds.field_names} HR:{train_ds.img_shape} {train_ds.factor}x')

In [ ]:
# Load FlowMatching model
from diffusionsr.runners.train_flow_matching import FlowMatchingModel
fm_model=FlowMatchingModel(
    results_folder=FM_RUN_DIR, lr_encoder_folder=FM_ENC_DIR if FM_ENCODING else None,
    train_dataset=train_ds, dev_dataset=dev_ds, test_dataset=test_ds,
    timesteps=TIMESTEPS, conditioning=FM_COND, encoding=FM_ENCODING,
    schedule=SCHEDULE, device=DEVICE, enc_output=False)
fm_model.load_saved_model()
print(f'FlowMatching loaded: {FM_RUN_DIR}')

In [ ]:
# Load DDPM model (LDM or DiffusionSR)
if MODEL2=='ldm':
    from diffusionsr.runners.train_ldm import LDMModel
    diff_model=LDMModel(
        vae_folder=VAE_DIR, results_folder=DIFF_RUN_DIR,
        lr_encoder_folder=DIFF_ENC_DIR if DIFF_ENCODING else None,
        train_dataset=train_ds, dev_dataset=dev_ds, test_dataset=test_ds,
        timesteps=TIMESTEPS, conditioning=DIFF_COND, encoding=DIFF_ENCODING,
        schedule=SCHEDULE, device=DEVICE, enc_output=False)
else:
    from diffusionsr.runners.train_diffusion import DiffusionModel
    diff_model=DiffusionModel(
        results_folder=DIFF_RUN_DIR,
        lr_encoder_folder=DIFF_ENC_DIR if DIFF_ENCODING else None,
        train_dataset=train_ds, dev_dataset=dev_ds, test_dataset=test_ds,
        timesteps=TIMESTEPS, conditioning=DIFF_COND, encoding=DIFF_ENCODING,
        schedule=SCHEDULE, device=DEVICE, enc_output=False)
diff_model.load_saved_model()
print(f'{MODEL2.upper()} loaded: {DIFF_RUN_DIR}')

In [ ]:
# ── Run step ablations on both models ─────────────────────────────────────────
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

fm_mae   = {n: [] for n in FM_EULER_STEPS}    # NFE = euler steps
fm_mpmae = {n: [] for n in FM_EULER_STEPS}
fm_time  = {n: [] for n in FM_EULER_STEPS}

dd_mae   = {s: [] for s in DDIM_SKIPS}        # NFE = TIMESTEPS/skip
dd_mpmae = {s: [] for s in DDIM_SKIPS}
dd_time  = {s: [] for s in DDIM_SKIPS}

for batch_i, batch in enumerate(test_dl):
    if batch_i >= N_ABL_BATCHES: break
    _, hr_b, lr_b, ul_b = batch[:4]

    # FM ablation
    if FM_ENCODING:
        fm_xe = fm_model.compute_x_e(lr_b, ul_b)
    else:
        fm_xe = None
    for ns in FM_EULER_STEPS:
        t0 = time.perf_counter()
        with torch.no_grad():
            _s = fm_model.batch_sample(dataset=test_ds, batch=hr_b.to(DEVICE), x_e=fm_xe, sampler='euler', n_steps=ns)
        fm_time[ns].append((time.perf_counter()-t0)/hr_b.shape[0])
        for si in range(hr_b.shape[0]):
            p = test_ds.unscale_data(_s[-1].cpu().numpy()[si], input_type='hr')
            g = test_ds.unscale_data(as_numpy(hr_b[si]), input_type='hr')
            fm_mae[ns].append(mae_rmse(p[ANALYSIS_CH], g[ANALYSIS_CH]))
            try:
                pmp,_ = get_profile(p[ANALYSIS_CH:ANALYSIS_CH+1])
                gmp,_ = get_profile(g[ANALYSIS_CH:ANALYSIS_CH+1])
                fm_mpmae[ns].append(float(np.mean(np.abs(pmp-gmp))))
            except: pass

    # DDIM ablation
    if DIFF_ENCODING:
        dd_xe = diff_model.compute_x_e(lr_b, ul_b)
    else:
        dd_xe = None
    for sk in DDIM_SKIPS:
        t0 = time.perf_counter()
        with torch.no_grad():
            _s = diff_model.batch_sample(dataset=test_ds, batch=hr_b.to(DEVICE), x_e=dd_xe, sampler='DDIM', skip=sk)
        dd_time[sk].append((time.perf_counter()-t0)/hr_b.shape[0])
        for si in range(hr_b.shape[0]):
            p = test_ds.unscale_data(_s[-1].cpu().numpy()[si], input_type='hr')
            g = test_ds.unscale_data(as_numpy(hr_b[si]), input_type='hr')
            dd_mae[sk].append(mae_rmse(p[ANALYSIS_CH], g[ANALYSIS_CH]))
            try:
                pmp,_ = get_profile(p[ANALYSIS_CH:ANALYSIS_CH+1])
                gmp,_ = get_profile(g[ANALYSIS_CH:ANALYSIS_CH+1])
                dd_mpmae[sk].append(float(np.mean(np.abs(pmp-gmp))))
            except: pass

print('Ablation complete.')
print(f'FM Euler steps={FM_EULER_STEPS}  ->  MAE={[f"{np.nanmean(fm_mae[n]):.4f}" for n in FM_EULER_STEPS]}')
print(f'DDIM skips={DDIM_SKIPS}          ->  MAE={[f"{np.nanmean(dd_mae[s]):.4f}" for s in DDIM_SKIPS]}')

In [ ]:
# ── NFE vs Quality comparison plot ────────────────────────────────────────────
fm_nfe  = FM_EULER_STEPS
dd_nfe  = [TIMESTEPS // sk for sk in DDIM_SKIPS]

fm_mae_v  = [np.nanmean(fm_mae[n])   for n in FM_EULER_STEPS]
fm_mp_v   = [np.nanmean(fm_mpmae[n]) if fm_mpmae[n] else float('nan') for n in FM_EULER_STEPS]
fm_time_v = [np.nanmean(fm_time[n])  for n in FM_EULER_STEPS]

dd_mae_v  = [np.nanmean(dd_mae[s])   for s in DDIM_SKIPS]
dd_mp_v   = [np.nanmean(dd_mpmae[s]) if dd_mpmae[s] else float('nan') for s in DDIM_SKIPS]
dd_time_v = [np.nanmean(dd_time[s])  for s in DDIM_SKIPS]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=130)

ax=axes[0]
ax.plot(fm_nfe, fm_mae_v, 'o-', color='darkorange', lw=2, ms=8, label='FlowMatching (Euler)')
ax.plot(dd_nfe, dd_mae_v, 's--', color='steelblue', lw=2, ms=8, label=f'{MODEL2.upper()} (DDIM)')
ax.set_xlabel('NFE (number of function evaluations)')
ax.set_ylabel('Field MAE (temperature)'); ax.set_title('Sample Quality vs. NFE')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

ax=axes[1]
ax.plot(fm_nfe, fm_mp_v, 'o-', color='darkorange', lw=2, ms=8, label='FlowMatching')
ax.plot(dd_nfe, dd_mp_v, 's--', color='steelblue', lw=2, ms=8, label=MODEL2.upper())
ax.set_xlabel('NFE'); ax.set_ylabel('MP-MAE (px)')
ax.set_title('Melt-Pool Depth Error vs. NFE'); ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

ax=axes[2]
ax.plot(fm_nfe, fm_time_v, 'o-', color='darkorange', lw=2, ms=8, label='FlowMatching')
ax.plot(dd_nfe, dd_time_v, 's--', color='steelblue', lw=2, ms=8, label=MODEL2.upper())
ax.set_xlabel('NFE'); ax.set_ylabel('Inference time per sample (s)')
ax.set_title('Inference Speed vs. NFE'); ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

plt.suptitle(
    f'FlowMatching (Euler) vs {MODEL2.upper()} (DDIM) — fields={FIELD_NAMES}\n'
    f'ablation_20260901 | enc_sdf | {N_ABL_BATCHES} batches of {BATCH_SIZE}',
    fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# ── Quality vs. wall-clock time scatter (shows the Pareto frontier) ────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=130)
for ax, fm_y, dd_y, ylabel in [
        (axes[0], fm_mae_v, dd_mae_v, 'Field MAE'),
        (axes[1], fm_mp_v,  dd_mp_v,  'MP-MAE (px)')]:
    sc1 = ax.scatter(fm_time_v, fm_y, c=fm_nfe, cmap='Oranges', s=120, zorder=5,
                     edgecolors='darkorange', linewidths=1.5, label='FlowMatching (Euler)')
    sc2 = ax.scatter(dd_time_v, dd_y, c=dd_nfe, cmap='Blues',   s=120, zorder=5,
                     edgecolors='steelblue',  linewidths=1.5, marker='s', label=f'{MODEL2.upper()} (DDIM)')
    ax.plot(fm_time_v, fm_y, '--', color='darkorange', lw=1.2, alpha=0.6)
    ax.plot(dd_time_v, dd_y, '--', color='steelblue',  lw=1.2, alpha=0.6)
    for i,ns in enumerate(fm_nfe): ax.annotate(f'{ns}', (fm_time_v[i], fm_y[i]), fontsize=8, color='darkorange', xytext=(4,3), textcoords='offset points')
    for i,sk in enumerate(DDIM_SKIPS): ax.annotate(f'skip{sk}', (dd_time_v[i], dd_y[i]), fontsize=8, color='steelblue', xytext=(4,-9), textcoords='offset points')
    ax.set_xlabel('Inference time per sample (s)'); ax.set_ylabel(ylabel)
    ax.set_title(f'{ylabel} vs. Time'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.suptitle('Quality vs. Inference Cost — FM (Euler, orange) vs. DDPM (DDIM, blue)\nNumbers = NFE / skip value', fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
rows = []
for ns in FM_EULER_STEPS:
    rows.append({'model':'FlowMatching','sampler':f'Euler n={ns}','nfe':ns,
                 'time_s':float(np.nanmean(fm_time[ns])),
                 'mae':float(np.nanmean(fm_mae[ns])),
                 'mp_mae':float(np.nanmean(fm_mpmae[ns])) if fm_mpmae[ns] else float('nan')})
for sk in DDIM_SKIPS:
    rows.append({'model':MODEL2,'sampler':f'DDIM skip={sk}','nfe':TIMESTEPS//sk,
                 'time_s':float(np.nanmean(dd_time[sk])),
                 'mae':float(np.nanmean(dd_mae[sk])),
                 'mp_mae':float(np.nanmean(dd_mpmae[sk])) if dd_mpmae[sk] else float('nan')})
df = pd.DataFrame(rows).sort_values('mae')
print('\nResults sorted by MAE (best first):')
print(df.to_string(index=False, float_format='{:.4f}'.format))

import os; os.makedirs(EVAL_OUT_DIR, exist_ok=True)
df.to_csv(f'{EVAL_OUT_DIR}/fm_vs_ddpm_steps_study.csv', index=False)
print(f'\nSaved -> {EVAL_OUT_DIR}/fm_vs_ddpm_steps_study.csv')